# RFM Segmentation Analysis
Synthetic e-commerce dataset, customer segmentation, revenue contribution, and churn risk analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

In [ ]:
orders = pd.read_csv('../data/orders_dataset.csv', parse_dates=['order_date'])
orders.head()

In [ ]:
orders.info()

In [ ]:
orders.isnull().sum()

In [ ]:
ref_date = pd.Timestamp('2026-06-01')
rfm = orders.groupby('user_id').agg({
    'order_date': lambda x: (ref_date - x.max()).days,
    'order_id': 'nunique',
    'amount': 'sum'
}).reset_index()
rfm.columns = ['user_id', 'recency', 'frequency', 'monetary']
rfm.head()

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_score']*100 + rfm['f_score']*10 + rfm['m_score']
rfm['segment'] = 'Other'
rfm.loc[rfm['rfm_score'] >= 444, 'segment'] = 'VIP'
rfm.loc[(rfm['r_score'] >= 3) & (rfm['f_score'] >= 3) & (rfm['segment'] == 'Other'), 'segment'] = 'Loyal'
rfm.loc[(rfm['r_score'] == 4) & (rfm['f_score'] <= 2) & (rfm['segment'] == 'Other'), 'segment'] = 'New'
rfm.loc[(rfm['r_score'] == 2) & (rfm['f_score'] >= 3) & (rfm['segment'] == 'Other'), 'segment'] = 'At Risk'
rfm.loc[(rfm['r_score'] <= 2) & (rfm['f_score'] <= 2) & (rfm['segment'] == 'Other'), 'segment'] = 'Dormant'
rfm['churn_risk'] = (rfm['recency'] > 90).astype(int)
rfm.head()

In [ ]:
segment_summary = rfm.groupby('segment').agg(
    users=('user_id', 'count'),
    revenue=('monetary', 'sum'),
    churn_users=('churn_risk', 'sum')
).reset_index()
segment_summary['revenue_share'] = segment_summary['revenue'] / segment_summary['revenue'].sum()
segment_summary['user_share'] = segment_summary['users'] / segment_summary['users'].sum()
segment_summary['churn_share_inside_segment'] = segment_summary['churn_users'] / segment_summary['users']
segment_summary.sort_values('revenue', ascending=False)

In [ ]:
plt.figure(figsize=(10,5))
plot_df = segment_summary.sort_values('revenue', ascending=False)
sns.barplot(data=plot_df, x='segment', y='revenue', palette='viridis')
plt.title('Revenue by Segment')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=segment_summary.sort_values('users', ascending=False), x='segment', y='users', palette='magma')
plt.title('Users by Segment')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Example conclusions
- VIP users are a small but strategically important segment with a disproportionately high revenue contribution.
- Loyal users form the core stable revenue base.
- At Risk and Dormant users represent the main reactivation opportunity.
- New users should receive onboarding and repeat-purchase стимулирование in the first 30–90 days.